# Live Stats Monitor

Professional live monitor for the Adaptive Quantum SOR API. Use the widget panel to choose the API endpoint and report options, then refresh a clean stats, policy, order-summary, and chart report.

## 1. Monitor Inputs

Use the controls below to choose the API URL, timeout, and report sections.

In [ ]:
from datetime import datetime, timezone
from html import escape
from IPython.display import HTML, display
import importlib
import pandas as pd

import adaptive_quantum_sor.client as sor_client_module

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None

sor_client_module = importlib.reload(sor_client_module)
SorNotebookClient = sor_client_module.SorNotebookClient

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

REPORT_CSS = """
<style>
.sor-report {font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color:#111827;}
.sor-hero {border:1px solid #d7dde6;border-radius:8px;padding:18px 20px;margin:4px 0 18px 0;background:linear-gradient(180deg,#ffffff 0%,#f8fafc 100%);}
.sor-hero h1 {font-size:24px;margin:0 0 6px 0;letter-spacing:0;}
.sor-hero p {margin:0;color:#5b6472;font-size:14px;}
.sor-section {margin:22px 0 10px 0;}
.sor-section h2 {font-size:18px;margin:0 0 4px 0;letter-spacing:0;}
.sor-section p {margin:0 0 8px 0;color:#5b6472;font-size:13px;}
.sor-kpi-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:10px 0 18px 0;}
.sor-kpi {border:1px solid #d7dde6;border-radius:8px;padding:12px 14px;background:#ffffff;}
.sor-kpi span {display:block;color:#5b6472;font-size:12px;margin-bottom:5px;}
.sor-kpi strong {display:block;color:#111827;font-size:22px;line-height:1.15;}
.sor-note {border-left:4px solid #1d4ed8;background:#eef5ff;padding:10px 12px;margin:10px 0 18px 0;color:#1e3a8a;}
.sor-badge {display:inline-block;border-radius:999px;padding:3px 9px;font-size:12px;font-weight:700;}
.sor-pass {background:#ecfdf5;color:#047857;border:1px solid #a7f3d0;}
.sor-warn {background:#fffbeb;color:#b45309;border:1px solid #fde68a;}
.sor-fail {background:#fef2f2;color:#b91c1c;border:1px solid #fecaca;}
table.sor-report-table {border-collapse:collapse;width:100%;table-layout:auto;margin:8px 0 18px 0;font-size:13px;}
table.sor-report-table th, table.sor-report-table td {border:1px solid #d7dde6;padding:7px 9px;vertical-align:top;white-space:normal;overflow-wrap:anywhere;word-break:break-word;max-width:560px;text-align:left;}
table.sor-report-table th {background:#f8fafc;color:#111827;font-weight:700;}
table.sor-report-table tbody tr:nth-child(even) {background:#fcfdff;}
.sor-bar-panel {border:1px solid #d7dde6;border-radius:8px;padding:12px;background:#ffffff;margin:8px 0 18px 0;}
.sor-bar-row {display:grid;grid-template-columns:220px 1fr 100px;gap:10px;align-items:center;margin:7px 0;}
.sor-bar-label {font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#374151;}
.sor-bar-track {height:12px;background:#eef2f7;border-radius:3px;overflow:hidden;}
.sor-bar {height:12px;background:#2563eb;}
.sor-bar-value {text-align:right;font-variant-numeric:tabular-nums;color:#111827;}
</style>
"""

def display_styles():
    display(HTML(REPORT_CSS))

def display_header(title, subtitle, generated_at):
    display(HTML(f"""
    <div class="sor-report sor-hero">
      <h1>{escape(title)}</h1>
      <p>{escape(subtitle)}</p>
      <p>Generated at {escape(generated_at)} UTC</p>
    </div>
    """))

def display_section(title, subtitle=""):
    subtitle_html = f"<p>{escape(subtitle)}</p>" if subtitle else ""
    display(HTML(f"<div class='sor-report sor-section'><h2>{escape(title)}</h2>{subtitle_html}</div>"))

def display_note(text):
    display(HTML(f"<div class='sor-report sor-note'>{escape(str(text))}</div>"))

def display_table(title, frame, subtitle=""):
    display_section(title, subtitle)
    if frame is None or frame.empty:
        display_note("No rows were returned for this section.")
        return
    html = frame.to_html(index=False, escape=True).replace('class="dataframe"', 'class="sor-report-table"')
    display(HTML("<div class='sor-report'>" + html + "</div>"))

def display_kpis(cards):
    html = ''.join(f"<div class='sor-kpi'><span>{escape(str(label))}</span><strong>{escape(str(value))}</strong></div>" for label, value in cards)
    display(HTML("<div class='sor-report sor-kpi-grid'>" + html + "</div>"))

def badge(label, tone):
    css = {"pass":"sor-pass", "warn":"sor-warn", "fail":"sor-fail"}.get(tone, "sor-warn")
    return f"<span class='sor-badge {css}'>{escape(str(label))}</span>"

def first_value(frame, *names, default=0):
    for name in names:
        if name in frame.columns and len(frame) > 0:
            return frame.iloc[0][name]
    return default


BASE_URL = "http://127.0.0.1:8080"
TIMEOUT_SECONDS = 5.0
INCLUDE_ORDER_SUMMARY = True
SHOW_NUMERIC_CHART = True

widget_controls = {}
if widgets is not None:
    widget_controls = {
        "base_url": widgets.Text(value=BASE_URL, description="API URL", layout=widgets.Layout(width="420px")),
        "timeout_seconds": widgets.FloatSlider(value=TIMEOUT_SECONDS, min=1.0, max=30.0, step=0.5, description="Timeout", continuous_update=False),
        "include_order_summary": widgets.Checkbox(value=INCLUDE_ORDER_SUMMARY, description="Order summary"),
        "show_numeric_chart": widgets.Checkbox(value=SHOW_NUMERIC_CHART, description="Numeric chart"),
    }
    display_styles()
    display(HTML("<div class='sor-report sor-section'><h2>Widget Control Panel</h2><p>Choose the monitoring view, then run the next cell to refresh the report.</p></div>"))
    display(widgets.VBox([
        widgets.HBox([widget_controls["base_url"], widget_controls["timeout_seconds"]]),
        widgets.HBox([widget_controls["include_order_summary"], widget_controls["show_numeric_chart"]]),
    ]))
else:
    display_styles()
    display_note("ipywidgets is not installed. Install python/requirements.txt to enable friendly controls; defaults will be used.")

def read_widget_inputs():
    if widget_controls:
        return {
            "base_url": widget_controls["base_url"].value,
            "timeout_seconds": float(widget_controls["timeout_seconds"].value),
            "include_order_summary": bool(widget_controls["include_order_summary"].value),
            "show_numeric_chart": bool(widget_controls["show_numeric_chart"].value),
        }
    return {
        "base_url": BASE_URL,
        "timeout_seconds": TIMEOUT_SECONDS,
        "include_order_summary": INCLUDE_ORDER_SUMMARY,
        "show_numeric_chart": SHOW_NUMERIC_CHART,
    }


## 2. Refresh Monitor Report

Run the next cell whenever you want a fresh snapshot.

In [ ]:
inputs = read_widget_inputs()
client = SorNotebookClient(base_url=inputs["base_url"], timeout_seconds=inputs["timeout_seconds"])
refreshed_at = datetime.now(timezone.utc)

stats = client.stats_dataframe()
policy = client.policy_dataframe()
snapshot = pd.concat([stats.add_prefix("stats."), policy.add_prefix("policy.")], axis=1)
order_summary_frames = client.order_summary_frames() if inputs["include_order_summary"] else {}

policy_version = first_value(policy, "policyVersion")
venue_count = first_value(stats, "venueCount")
parent_orders = first_value(stats, "parentOrderCount", "ordersAccepted")
child_orders = first_value(stats, "childOrderCount", "ordersRouted")
run_tone = "pass" if int(venue_count or 0) > 0 else "warn"

display_styles()
display_header(
    "Live SOR Monitor",
    f"Snapshot from {inputs['base_url']}",
    refreshed_at.strftime("%Y-%m-%d %H:%M:%S"),
)
display(HTML(f"<div class='sor-report'>{badge('Live snapshot', run_tone)}</div>"))
display_kpis([
    ("Venue count", venue_count),
    ("Policy version", policy_version),
    ("Parent orders", parent_orders),
    ("Child orders", child_orders),
])

display_table("1. Snapshot Manifest", pd.DataFrame([inputs | {"refreshedAtUtc": refreshed_at.strftime("%Y-%m-%d %H:%M:%S")}]), "Controls used for this refresh.")
display_table("2. Combined Stats And Policy", snapshot, "One-row view combining /stats/current and /policy/current.")
display_table("3. Stats Response", stats, "Raw stats fields returned by /stats/current.")
display_table("4. Policy Response", policy, "Raw active policy identity returned by /policy/current.")

if order_summary_frames:
    for title, frame in order_summary_frames.items():
        display_table(f"5. Order Summary - {title}", frame, "Cumulative order routing totals from /orders/summary.")

if inputs["show_numeric_chart"]:
    chart_rows = []
    for frame_name, frame in [("stats", stats), ("policy", policy)]:
        for column in frame.columns:
            value = frame.iloc[0][column]
            if pd.api.types.is_number(value) and not isinstance(value, bool) and pd.notna(value):
                chart_rows.append({"source": frame_name, "metric": column, "value": float(value)})
    chart = pd.DataFrame(chart_rows, columns=["source", "metric", "value"]).sort_values(["source", "metric"]).reset_index(drop=True)
    display_section("6. Numeric Metric Chart", "Scaled bars for all numeric stats and policy fields.")
    if chart.empty:
        display_note("No numeric stats are available for charting yet.")
    else:
        scale = max(chart["value"].abs().max(), 1.0)
        chart["bar_width_pct"] = (chart["value"].abs() / scale * 100).round(1)
        bars = ''.join(
            f"<div class='sor-bar-row'><span class='sor-bar-label'>{escape(str(row.source))}.{escape(str(row.metric))}</span>"
            f"<div class='sor-bar-track'><div class='sor-bar' style='width:{row.bar_width_pct}%'></div></div>"
            f"<span class='sor-bar-value'>{row.value:g}</span></div>"
            for row in chart.itertuples(index=False)
        )
        display(HTML(f"<div class='sor-report sor-bar-panel'>{bars}</div>"))
        display_table("Numeric Metric Data", chart[["source", "metric", "value"]])
